# Cheatsheet: osobine expectationa

Notebook za brzo ponavljanje ocekivanja `E[X]`, varijance, kovarijance, uslovnog ocekivanja i najcescih trikova. Korisno za RL, MDP-e, bandite, DP i probability zadatke.

In [ ]:
import numpy as np
from collections import defaultdict

np.set_printoptions(precision=4, suppress=True)

## Definicije

Diskretno:

`E[X] = sum_x x p(x)`

Za funkciju random varijable:

`E[f(X)] = sum_x f(x) p(x)`

Kontinuirano:

`E[X] = int x f_X(x) dx`

`E[g(X)] = int g(x) f_X(x) dx`

U kodu, ako je distribucija dict `{value: probability}`:

```python
EX = sum(x * p for x, p in P.items())
EgX = sum(g(x) * p for x, p in P.items())
```

In [ ]:
P = {0: 0.2, 1: 0.5, 3: 0.3}
EX = sum(x*p for x, p in P.items())
EX2 = sum(x**2*p for x, p in P.items())
print("E[X]  =", EX)
print("E[X^2]=", EX2)

## Linearitet expectationa

Najvaznija osobina:

`E[aX + bY + c] = aE[X] + bE[Y] + c`

Ovo vrijedi uvijek, cak i kad `X` i `Y` nisu nezavisni.

Posebno:

- `E[X + Y] = E[X] + E[Y]`
- `E[cX] = cE[X]`
- `E[c] = c`
- `E[sum_i X_i] = sum_i E[X_i]`

RL primjer:

`E[R + gamma V(S')] = E[R] + gamma E[V(S')]`.

## Produkti i nezavisnost

Ako su `X` i `Y` nezavisni:

`E[XY] = E[X]E[Y]`

Ako nisu nezavisni, ovo generalno nije tacno.

Korisno pravilo:

`Cov(X,Y) = E[XY] - E[X]E[Y]`

Dakle:

`E[XY] = Cov(X,Y) + E[X]E[Y]`.

In [ ]:
# Zavisni primjer: Y = X, pa E[XY] = E[X^2], ne E[X]^2.
P = {-1: 0.5, 1: 0.5}
EX = sum(x*p for x, p in P.items())
EXY = sum((x*x)*p for x, p in P.items())
print("E[X]E[Y] =", EX*EX)
print("E[XY]    =", EXY)

## Varijanca i standardna devijacija

Definicija:

`Var(X) = E[(X - E[X])^2]`

Najkorisniji racunski oblik:

`Var(X) = E[X^2] - E[X]^2`

Skaliranje:

`Var(aX + b) = a^2 Var(X)`

Za sumu:

`Var(X+Y) = Var(X) + Var(Y) + 2Cov(X,Y)`

Ako su nezavisni:

`Var(X+Y) = Var(X) + Var(Y)`.

In [ ]:
P = {0: 0.2, 1: 0.5, 3: 0.3}
EX = sum(x*p for x, p in P.items())
EX2 = sum(x*x*p for x, p in P.items())
VarX = EX2 - EX**2
print("E[X]  =", EX)
print("Var X =", VarX)
print("Std X =", np.sqrt(VarX))

## Uslovno ocekivanje

Diskretno:

`E[X | Y=y] = sum_x x p(x | y)`

Law of total expectation:

`E[X] = E[ E[X | Y] ]`

Diskretno po `Y`:

`E[X] = sum_y E[X | Y=y] p(y)`

RL/MDP oblik:

`E[R + gamma V(S') | S=s, A=a] = sum_{s',r} p(s',r|s,a) [r + gamma V(s')]`.

In [ ]:
# MDP/Bellman expected value from a dictionary p(s', r | s, a)
Psr = {("good", 1): 0.7, ("bad", -1): 0.3}
V = {"good": 10, "bad": 2}
gamma = 0.9

q_sa = sum(p * (r + gamma * V[sp]) for (sp, r), p in Psr.items())
print("E[R + gamma V(S') | s,a] =", q_sa)

## Indicator random variables

Ako je `I_A` indikator dogadjaja `A`:

`I_A = 1` ako se `A` desi, inace `0`.

Tada:

`E[I_A] = P(A)`

Ovo je super trik za brojanje: ako je `N = sum_i I_i`, onda

`E[N] = sum_i P(I_i=1)`.

## Jensenova nejednakost

Ako je `phi` konveksna:

`phi(E[X]) <= E[phi(X)]`

Ako je `phi` konkavna, znak ide obrnuto.

Primjeri:

- `x^2` je konveksna: `(E[X])^2 <= E[X^2]`
- `log(x)` je konkavna: `E[log X] <= log(E[X])`

## Ceste distribucije

| distribucija | mean | variance |
|---|---:|---:|
| Bernoulli(`p`) | `p` | `p(1-p)` |
| Binomial(`n,p`) | `np` | `np(1-p)` |
| Geometric(`p`) broj pokusaja do uspjeha | `1/p` | `(1-p)/p^2` |
| Poisson(`lambda`) | `lambda` | `lambda` |
| Uniform discrete `{1,...,n}` | `(n+1)/2` | `(n^2-1)/12` |
| Uniform continuous `[a,b]` | `(a+b)/2` | `(b-a)^2/12` |
| Normal `N(mu, sigma^2)` | `mu` | `sigma^2` |
| Exponential(`lambda`) | `1/lambda` | `1/lambda^2` |

## RL/DP cheat formulas

DP expected cost backup:

`J_k(x) = min_u sum_w P(w|x,u,k) [g_k(x,u,w) + J_{k+1}(f_k(x,u,w))]`

Bellman expectation for a fixed policy:

`v_pi(s) = sum_a pi(a|s) sum_{s',r} p(s',r|s,a) [r + gamma v_pi(s')]`

Bellman optimality:

`v*(s) = max_a sum_{s',r} p(s',r|s,a) [r + gamma v*(s')]`

TD target is an empirical/bootstrapped expectation sample:

`target = R + gamma V(S')`.

## Quick checklist

1. Linearity does not need independence.
2. Product rule `E[XY]=E[X]E[Y]` needs independence or zero covariance.
3. Always compute variance as `E[X^2]-E[X]^2` when possible.
4. Conditional expectation is itself a random variable before you average it.
5. In MDPs, expectation is over `(S', R)` given `(S, A)`.
6. Terminal next-state value is usually `0`.